# Section 4 · Notebook 3 — Graph Pattern: Order Processing

Orders follow explicit, auditable paths:

```
                    ┌─ HIGH_VALUE ──→ Manual Review ─┐
Risk Analyzer ──────┼─ SUSPICIOUS ──→ Fraud Check ───┼──→ Inventory ──┬─ IN_STOCK ─────→ Shipping
                    └─ LOW_RISK ─────────────────────┘               └─ OUT_OF_STOCK ─→ Refund
```

Unlike a swarm, the routing lives in code. Every path is visible and can be audited.

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: /private/tmp/scaler-code-venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from strands.models.bedrock import BedrockModel

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "amazon.nova-lite-v1:0")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Using {MODEL_ID} in {REGION}")

Using amazon.nova-lite-v1:0 in us-east-1


In [3]:
from strands import Agent
from strands.multiagent import GraphBuilder

## Specialised agents

The two "decision" agents end their answer with a fixed line (`RISK_CATEGORY: ...`, `STOCK_STATUS: ...`). The routing functions read only that line, so an explanation that merely *mentions* another category cannot send the order down the wrong path.

In [4]:
risk_analyzer = Agent(
    name="risk_analyzer",
    model=model,
    callback_handler=None,
    system_prompt="""Analyze the order's risk level and pick exactly ONE category:
- HIGH_VALUE: orders over $1,000
- SUSPICIOUS: mismatched billing/shipping country, brand-new account with a large order, or known fraud patterns
- LOW_RISK: regular orders from trusted customers
If more than one applies, SUSPICIOUS wins over HIGH_VALUE.
Give a one-paragraph explanation, then finish with a final line in exactly this format:
RISK_CATEGORY: <HIGH_VALUE|SUSPICIOUS|LOW_RISK>""",
)

manual_review_agent = Agent(
    name="manual_review_agent",
    model=model,
    callback_handler=None,
    system_prompt="Review high-value orders. Verify customer identity, check order history, and approve or flag for further review.",
)

fraud_detection_agent = Agent(
    name="fraud_detection_agent",
    model=model,
    callback_handler=None,
    system_prompt="Run fraud detection on suspicious orders: known fraud patterns, velocity checks, and device fingerprinting. Decide whether to proceed.",
)

inventory_agent = Agent(
    name="inventory_agent",
    model=model,
    callback_handler=None,
    system_prompt="""Check inventory for the order items. Assume laptops, books and phones are in stock and
anything described as 'limited edition' or 'pre-order' is out of stock.
Explain briefly, then finish with a final line in exactly this format:
STOCK_STATUS: <IN_STOCK|OUT_OF_STOCK>""",
)

shipping_agent = Agent(
    name="shipping_agent",
    model=model,
    callback_handler=None,
    system_prompt="Generate a shipping label and process the shipment. Confirm a tracking number and estimated delivery date.",
)

refund_agent = Agent(
    name="refund_agent",
    model=model,
    callback_handler=None,
    system_prompt="Process a refund for out-of-stock items. Notify the customer and initiate the payment reversal.",
)

print("✓ All agents created")

✓ All agents created


## Routing conditions

Each condition reads the previous node's output and returns True/False. The logic is plain Python — not hidden inside model reasoning.

In [5]:
import re


def decision(state, node_id: str, label: str) -> str:
    """Return the value of the final 'LABEL: VALUE' line written by a node."""
    text = str(state.results.get(node_id, ""))
    matches = re.findall(rf"{label}\s*:\s*([A-Z_]+)", text.upper())
    return matches[-1] if matches else ""


def is_high_value(state):
    return decision(state, "risk_analysis", "RISK_CATEGORY") == "HIGH_VALUE"


def is_suspicious(state):
    return decision(state, "risk_analysis", "RISK_CATEGORY") == "SUSPICIOUS"


def is_low_risk(state):
    return decision(state, "risk_analysis", "RISK_CATEGORY") == "LOW_RISK"


def is_in_stock(state):
    return decision(state, "inventory_check", "STOCK_STATUS") == "IN_STOCK"


def is_out_of_stock(state):
    return decision(state, "inventory_check", "STOCK_STATUS") == "OUT_OF_STOCK"


print("✓ Routing conditions defined")

✓ Routing conditions defined


## Build the graph

In [6]:
builder = GraphBuilder()

# Nodes
builder.add_node(risk_analyzer, "risk_analysis")
builder.add_node(manual_review_agent, "manual_review")
builder.add_node(fraud_detection_agent, "fraud_check")
builder.add_node(inventory_agent, "inventory_check")
builder.add_node(shipping_agent, "shipping")
builder.add_node(refund_agent, "refund")

# Conditional edges out of risk analysis
builder.add_edge("risk_analysis", "manual_review", condition=is_high_value)
builder.add_edge("risk_analysis", "fraud_check", condition=is_suspicious)
builder.add_edge("risk_analysis", "inventory_check", condition=is_low_risk)

# Review / fraud check → inventory
builder.add_edge("manual_review", "inventory_check")
builder.add_edge("fraud_check", "inventory_check")

# Inventory → ship or refund
builder.add_edge("inventory_check", "shipping", condition=is_in_stock)
builder.add_edge("inventory_check", "refund", condition=is_out_of_stock)

builder.set_entry_point("risk_analysis")
graph = builder.build()
print("✓ Graph built")


def show_path(result):
    path = " → ".join(node.node_id for node in result.execution_order)
    print(f"Status: {result.status}\nPath:   {path}")
    last = result.execution_order[-1].node_id
    print(f"\nFinal step ({last}):\n{result.results[last].result}")

✓ Graph built


/var/folders/dr/4p4pb8091bn4xdxrt591sc6m0000gn/T/ipykernel_14151/3216984768.py:25: UserWarning: graph has no max_node_executions or execution_timeout set; execution is unbounded
  graph = builder.build()


## Process orders — each one takes a different path

In [7]:
# High value → manual review → inventory → shipping
result = graph("Order #12345: $1,500 laptop from an existing customer with 3 previous orders; billing and shipping address match")
show_path(result)

Status: Status.COMPLETED
Path:   risk_analysis → inventory_check → shipping

Final step (shipping):
### Shipping Label Generation

**Order Details:**
- **Order Number:** 12345
- **Customer:** Existing customer with 3 previous orders
- **Item:** $1,500 Laptop
- **Billing Address:** [Customer's Billing Address]
- **Shipping Address:** [Customer's Shipping Address]

**Shipping Label:**
- **Recipient:** [Customer's Full Name]
- **Address:** [Customer's Shipping Address]
- **City:** [City]
- **State:** [State]
- **ZIP Code:** [ZIP Code]
- **Country:** [Country]
- **Phone Number:** [Customer's Contact Number]

**Package Contents:**
- **Item:** $1,500 Laptop
- **Quantity:** 1
- **Weight:** [Weight of the Laptop]
- **Value:** $1,500

### Process Shipment

1. **Print Shipping Label:**
    - The shipping label has been generated and attached to the package.

2. **Prepare Package for Shipment:**
    - The laptop has been carefully packed to ensure safe transit.
    - The package has been sealed a

In [8]:
# Low risk → inventory → shipping
result = graph("Order #67890: $45 book from a returning customer with 50+ previous orders")
show_path(result)

Status: Status.COMPLETED
Path:   risk_analysis → inventory_check → shipping

Final step (shipping):
### Shipping Label Generation

**Order Details:**
- **Order Number:** 67890
- **Customer:** Returning customer with 50+ previous orders
- **Item:** $45 Book
- **Billing Address:** [Customer's Billing Address]
- **Shipping Address:** [Customer's Shipping Address]

**Shipping Label:**
- **Recipient:** [Customer's Full Name]
- **Address:** [Customer's Shipping Address]
- **City:** [City]
- **State:** [State]
- **ZIP Code:** [ZIP Code]
- **Country:** [Country]
- **Phone Number:** [Customer's Contact Number]

**Package Contents:**
- **Item:** $45 Book
- **Quantity:** 1
- **Weight:** [Weight of the Book]
- **Value:** $45

### Process Shipment

1. **Print Shipping Label:**
    - The shipping label has been generated and attached to the package.

2. **Prepare Package for Shipment:**
    - The book has been carefully packed to ensure safe transit.
    - The package has been sealed and ready for s

In [9]:
# Suspicious → fraud check → inventory → refund (limited edition is out of stock)
result = graph(
    "Order #11111: $800 limited edition headphones, account created 5 minutes ago, "
    "shipping to a different country than the billing address"
)
show_path(result)

Status: Status.COMPLETED
Path:   risk_analysis → fraud_check → inventory_check → refund

Final step (refund):
### Process Refund for Out-of-Stock Items

#### Steps to Process Refund and Notify Customer

1. **Verify the Out-of-Stock Status:**
   - Confirm that the limited edition headphones (Order #11111) are indeed out of stock.
   - **Stock Status:** OUT_OF_STOCK

2. **Notify the Customer:**
   - Draft an email and/or SMS to inform the customer that their order for the limited edition headphones cannot be fulfilled due to the item being out of stock.
   - Ensure the message is clear, concise, and apologetic for any inconvenience caused.

#### Sample Notification Message:

**Email:**

Subject: Order #11111 - Update on Your Limited Edition Headphones

Dear [Customer Name],

We hope this message finds you well. We are writing to inform you that the limited edition headphones you ordered (Order #11111) are currently out of stock.

Unfortunately, due to their limited availability, we are u

## Key takeaways

1. **Explicit routing** — every path is visible in code
2. **Auditable** — "how do you handle fraud?" → point at the graph
3. **Acyclic** — edges only point forward, so every order reaches an end state
4. **Use it when** you have an established procedure (compliance, approvals, order processing)
5. **Don't use it when** the path can't be predicted up front — use a swarm instead